# Block analysis report — template

**Copy this notebook into your analysis project as `report.ipynb`** alongside your block subpackage, then change `BLOCK_NAME`, `MODULES`, and `TARGETS` to match your block. Everything else is renderer code that walks the project's results and produces the design-review HTML.

**What this notebook renders:**

| Section | Source |
|---|---|
| Project-wide summary | `framework.project_report_html` |
| Block-focused report (contracts, verifications, key quantities, cross-block consumers) | `framework.block_report_html` |
| Per-Quantity drill-down with provenance chains | `framework.quantity_to_html` |

All renderers consume the same `results` and `test_results` dicts — one source of truth, two surfaces (raw HTML string, or `IPython.display.HTML` wrapper).

## 1. Project, modules, targets

Adapt to your block. The pattern is: load the project, import the block's `leaves` / `analysis` / `contracts` / `verifications` modules, and list the headline DAG nodes you want the report to surface. The renderers also need the contracts' `compares_to` actuals and Quantity-valued `assumed_inputs`; pass `return_all_computed=True` to `project.run` (section 2) and those are included automatically — you don't have to enumerate them in `TARGETS`.

In [ ]:
import sys
from pathlib import Path

# Adapt these three lines for your project's directory layout.
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT.parent))  # for shared components/

from framework import Project, run_verifications

# === Customise these for your block ===
BLOCK_NAME = "can_transceiver"

from blocks.can_transceiver import leaves, analysis, contracts, verifications
from project.requirements import CAN_5V_RAIL  # adapt to your project

MODULES = [leaves, analysis, contracts, verifications]
TARGETS = [
    # The headline values you want surfaced in Key Quantities. Contracts +
    # their compares_to + assumed_inputs are pulled in automatically by
    # return_all_computed=True below.
    "power",
    "thermal_rise",
    "t_j",
    "t_j_max",
]
RUN_INPUTS = {"can_5v_rail": CAN_5V_RAIL}
# === End customisation ===

project = Project.load(
    scenarios=PROJECT_ROOT / "project" / "scenarios.toml",
    modes=PROJECT_ROOT / "project" / "modes.toml",
)

## 2. Run the DAG + verifications

`project.run(..., return_all_computed=True)` returns the full set of nodes Hamilton computed for this invocation — including contracts, their `compares_to` actuals, and any Quantity-valued `assumed_inputs` keys that were auto-augmented for the consistency checks. The renderers walk this full dict to produce the declared-vs-actual columns. Verification tests run separately via `run_verifications`.

In [ ]:
results = project.run(
    modules=MODULES,
    targets=TARGETS,
    inputs=RUN_INPUTS,
    return_all_computed=True,
)

test_results = run_verifications(MODULES, results)
print(f"DAG nodes computed: {sorted(results)}")
print(f"Verification tests: {len(test_results)}")

## 3. Project-wide report

Header counts (blocks / contracts / verifications, pass/fail/warn/info breakdown) + global verification roll-up + per-block subsections + requirement-coverage matrix. Use this for the system-wide design-review view; use the block-focused view in section 4 for deeper drill-down on a single block.

In [ ]:
from framework import display_project_report

display_project_report(
    project,
    modules=MODULES,
    results=results,
    test_results=test_results,
)

## 4. Block-focused report

Same renderer machinery, scoped to one block: contracts table (declared vs actual per mode, with status badges), verifications filtered to this block's tests, and every block-owned Quantity rendered via `quantity_to_html` with its provenance chain in a collapsible. The `Consumed by` section under each Contract is auto-populated by walking the provenance graph forward from the Contract node.

In [ ]:
from framework import display_block_report

display_block_report(
    project,
    modules=MODULES,
    results=results,
    block=BLOCK_NAME,
    test_results=test_results,
)

## 5. Quantity drill-down

Single-Quantity view with the provenance chain shown by default. Use this for design-review deep-dives on the values you want reviewers to focus on (the headline failure, the binding bound, the closest-to-spec margin). `quantity_to_html` returns an HTML string if you need to embed it elsewhere; `display_quantity` wraps it for inline notebook display.

In [ ]:
from framework import display_quantity

# Customise: pick the Quantity (or several) you want highlighted.
display_quantity(results["t_j"], name="Junction temperature (t_j)")

## 6. Export

Convert this notebook to a self-contained HTML deliverable:

```powershell
../hw_analysis_framework/.venv/Scripts/python.exe -m jupyter nbconvert --to html --execute report.ipynb
```

PDF rendering (design doc §12 mentions WeasyPrint) is deferred to step 12d — for now, the browser's print-to-PDF on the executed HTML is the recommended path.